# Figure 6: X inactivation, copy by copy, in GM12878 and IMR-90

This notebook reproduces Fig. 6a–i from the per-cell BAM files of the 629 human nuclei (469 GM12878, a clonal line whose inactive X is the paternal copy, and 160 IMR-90, a mosaic line):

| Panel | What | Made in |
|---|---|---|
| a | schematic of the allele assignment | Step 8 |
| b, c, e, f (right), g | each X copy's contacts: maps, P(s), contacts ≥1 Mb, compartments, DXZ4 crossing; escape and inactivated genes; XCI-strength quartiles | Step 5 |
| d | Xi − Xa methylation and accessibility at X-linked elements, and their contacts | Steps 3, 5 |
| f (left) | escape against inactivated genes' levels | Step 5 |
| h | chrX's excess concordance over the autosomes, observed and expected from the copies' level difference | Steps 4, 6 |
| i | concordance on each copy against same-cell, same-copy controls | Step 7 |

Each command is skipped when its outputs already exist, so the notebook can be re-run, or resumed after an interruption. Set `DRY_RUN = True` to print the commands only. See `README.md` (this folder) for installation, data download and compute requirements.

## Configuration

Paths come from `paths.yaml` at the repository root; edit that, not this notebook. The per-cell BAM files are read where the pipeline wrote them (`04.alignment_snakemake/` under `<work_root>/gm12878_cellline/` and `<work_root>/mixture_cellline/`), meth_concord is the checkout at `<software_root>/meth_concord`, and this analysis keeps the public data (`annotations/`, made by `download_public_data.sh`) and its results (`results/gm12878_imr90/`) under `<work_root>/concordance/`.

In [ ]:
# --- environment ---------------------------------------------------------------
# This notebook was written against the `meth_concord` conda environment, made
# as in "1. Install" of this folder's README.md.
# See env/README.md for which notebook uses which environment.
_ENV_EXPECTED = "meth_concord"
import os as _os, sys as _sys
if _os.path.basename(_sys.prefix) != _ENV_EXPECTED:
    print(f"[env] expected the '{_ENV_EXPECTED}' environment, running in "
          f"'{_os.path.basename(_sys.prefix)}' ({_sys.prefix}). See env/README.md.",
          file=_sys.stderr)

# --- repo paths: edit paths.yaml at the repository root, not this notebook ---
if "REPO_ROOT" not in globals():
    _r = _os.getcwd()
    while _r != "/" and not _os.path.exists(_os.path.join(_r, "paths.py")):
        _r = _os.path.dirname(_r)
    assert _os.path.exists(_os.path.join(_r, "paths.py")), \
        "paths.py not found above the working directory -- start Jupyter inside the repository"
    REPO_ROOT = _r
_sys.path.insert(0, REPO_ROOT)
from paths import SOFTWARE, WORK

import sys
from pathlib import Path

HERE = Path(REPO_ROOT) / "main_figures" / "fig5_fig6"   # this folder
METH_CONCORD = Path(SOFTWARE) / "meth_concord"          # software/README.md
BAMS = Path(WORK)                   # the sample sheet's BAM paths start here
DATA = Path(WORK) / "concordance"   # the public data and the results
ANNOT = DATA / "annotations"        # download_public_data.sh ANNOT hg38
OUT = DATA / "results" / "gm12878_imr90"
THREADS = 10
DRY_RUN = False                     # True: print the commands only

In [ ]:
sys.path.insert(0, str(HERE))
import shlex
import shutil
import pandas as pd
from repro import compare, run, show

PY = sys.executable
S = METH_CONCORD / "scripts"
HG38 = ANNOT / "hg38"
XA = HG38 / "xci_annotations"
FASTA = HG38 / "GCA_000001405.15_GRCh38_no_alt_analysis_set.fa"
TWOBIT = FASTA.with_suffix(".2bit")
ISLANDS = HG38 / "UCSC.cpgIsland.hg38.sort.bed"
ELEMENTS = {"promoter": HG38 / "gm12878_promoter.bed",
            "enhancer": HG38 / "gm12878_enhancer.bed",
            "ctcf": HG38 / "gm12878_ctcf.bed"}
CONTEXTS = (f"GCH:{HG38 / 'GCH_motif_seqkit.hg38.bed.gz'},"
            f"HCG:{HG38 / 'HCG_motif_seqkit.hg38.bed.gz'}")
LOGS, FIG = OUT / "logs", OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

sheet = pd.read_csv(HERE / "samples/gm12878_imr90.samples.tsv", sep="\t")
sheet["bam_path"] = [str(BAMS / p) for p in sheet["bam_path"]]
SAMPLES = OUT / "gm12878_imr90.samples.tsv"
sheet.to_csv(SAMPLES, sep="\t", index=False)
missing = [p for p in sheet["bam_path"] if not Path(p).exists()]
print(f"{len(sheet):,} cells:", sheet["group"].value_counts().to_dict())
print(f"{len(missing):,} BAM files missing" + (f", e.g. {missing[0]}" if missing else ""))
if not (S / "figure6_xci.py").exists():
    print(f"meth_concord not found at {METH_CONCORD}: see software/README.md")

## Step 1. Phased heterozygous SNVs

GM12878 (NA12878): the 1000 Genomes high-coverage phased panel on GRCh38, chrX phased by transmission from the parents (NA12891, NA12892), so hap1 is the paternal copy. The panel is ~32 GB; each chromosome is downloaded, md5-checked, reduced to NA12878's sites and deleted. IMR-90: the SNVs of its haplotype-resolved assemblies against T2T-CHM13v2.0 (figshare), lifted to GRCh38 with UCSC's `hs1ToHg38` chain and checked against the GRCh38 sequence.

In [ ]:
SNP = OUT / "snps"
VCF = {"GM12878": SNP / "NA12878.het.vcf.gz", "IMR90": SNP / "IMR90.het.vcf.gz"}
run([PY, S / "phased_snps.py", "1kgp", "--sample", "NA12878",
     "--output", VCF["GM12878"], "--work-dir", SNP / "tmp_1kgp", "--n-jobs", THREADS],
    outputs=[VCF["GM12878"]], log=LOGS / "snps_NA12878.log", dry_run=DRY_RUN)
run([PY, S / "phased_snps.py", "syri", "--fasta", FASTA,
     "--output", VCF["IMR90"], "--work-dir", SNP / "tmp_imr90"],
    outputs=[VCF["IMR90"]], inputs=[FASTA], log=LOGS / "snps_IMR90.log",
    dry_run=DRY_RUN)

## Step 2. End posteriors

The pipeline's steps 1–4 as in the cortex notebook, on GRCh38, in two passes so that X is scored under the autosomes' model: the beta-binomial mixture is fitted on the autosomes (the pipeline leaves chrX and chrY out by default; `pipeline_autosomes`), then steps 2 and 4 are rerun on every chromosome (`--exclude-chrom none`) with that fit (`pipeline`). The autosomes' posteriors are the same in both; the second pass re-reads the BAM files.

In [ ]:
PIPE0 = OUT / "pipeline_autosomes"   # pass 1: the mixture fit, autosomes only
PIPE = OUT / "pipeline"              # pass 2: every chromosome, under that fit
for ctx in ("GCH", "HCG"):
    bed = HG38 / f"{ctx}_motif_seqkit.hg38.bed.gz"
    base = [PY, "-m", "nome_hic_concordance", "--samples", SAMPLES,
            "--contexts", ctx, "--context-bed", f"{ctx}:{bed}",
            "--reference", TWOBIT, "--n-threads", THREADS]
    fit0, fit = (d / f"context_{ctx}" / "03_mixture_model" / "em_fit_all.json"
                 for d in (PIPE0, PIPE))
    if not fit.exists():                     # pass 2 not yet given its fit
        run([*base, "--output-dir", PIPE0, "--steps", "1,2,3"],
            outputs=[fit0], inputs=[SAMPLES, bed, TWOBIT],
            log=LOGS / f"pipeline_autosomes_{ctx}.log", dry_run=DRY_RUN)
        if fit0.exists():
            fit.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(fit0, fit)
    run([*base, "--output-dir", PIPE, "--exclude-chrom", "none", "--steps", "2,4"],
        outputs=[PIPE / f"context_{ctx}" / "04_posteriors"], inputs=[fit],
        log=LOGS / f"pipeline_{ctx}.log", dry_run=DRY_RUN)

POSTERIORS = ["--posteriors", f"GCH={PIPE / 'context_GCH' / '04_posteriors'}",
              "--posteriors", f"HCG={PIPE / 'context_HCG' / '04_posteriors'}"]

## Step 3. Each read pair on its copy; each copy's levels

`allelic_imbalance.py` tags every read pair that covers a phased SNV with its haplotype (`haplotypes/<cell>.haplotypes.parquet`), and summarises each copy's methylation and accessibility: first for both lines, which also calls each IMR-90 cell's inactive X (`allelic_cells.tsv`), then for GM12878 at CpG islands and its ENCODE promoters, enhancers and CTCF sites (Fig. 6d, top).

In [ ]:
HAP = OUT / "haplotypes"
PHASED = OUT / "allelic_phased"
ELEM = OUT / "allelic_elements"
run([PY, S / "allelic_imbalance.py", "--samples", SAMPLES,
     "--vcf", f"GM12878={VCF['GM12878']}", "--vcf", f"IMR90={VCF['IMR90']}",
     "--context-bed", CONTEXTS, "--region", f"island={ISLANDS}", "--mates",
     "--haplotype-dir", HAP, "--n-threads", THREADS, "--output-dir", PHASED],
    outputs=[HAP, PHASED / "allelic_cells.tsv"], inputs=[*VCF.values(), ISLANDS],
    log=LOGS / "allelic_phased.log", dry_run=DRY_RUN)
run([PY, S / "allelic_imbalance.py", "--samples", SAMPLES,
     "--vcf", f"GM12878={VCF['GM12878']}", "--context-bed", CONTEXTS,
     "--region", f"island={ISLANDS}",
     *[a for k, p in ELEMENTS.items() for a in ("--region", f"{k}={p}")],
     "--mates", "--n-threads", THREADS, "--output-dir", ELEM],
    outputs=[ELEM / "allelic_summary.tsv", ELEM / "allelic_snps.tsv"],
    inputs=[VCF["GM12878"], ISLANDS, *ELEMENTS.values()],
    log=LOGS / "allelic_elements.log", dry_run=DRY_RUN)

## Step 4. Concordance on chrX, copy by copy

`annotation_class_concordance.py --chrx --haplotypes` measures chrX's concordance against the autosomes' and, with the haplotype tags, each copy's, in sliding windows along X (Fig. 6h and the tracks of Fig. 6b). The run also classes the autosomes' contacts by 4DN replication timing and lamina association (`download_public_data.sh` fetches both), as the published run did.

In [ ]:
CHRX = OUT / "chrx_classes"
flags = [w for line in (HG38 / "rt_4dn" / "signal_track_flags.txt").read_text().splitlines()
         if not line.startswith("#") for w in shlex.split(line)] \
    if (HG38 / "rt_4dn" / "signal_track_flags.txt").exists() else []
rt_log2 = ["--rt-log2" if w == "--signal-track" else w for w in flags]
LAD, RTC = HG38 / "lad_4dn", HG38 / "rt_4dn"
run([PY, S / "annotation_class_concordance.py", *POSTERIORS,
     "--samples", SAMPLES, "--group-column", "group", "--batch-column", "batch",
     "--genome", "hg38", "--cache-dir", ANNOT / "cache", *rt_log2,
     "--annotation", f"LAD={LAD / 'cLAD.bed'},{LAD / 'fLAD.bed'},{LAD / 'ciLAD.bed'}",
     "--annotation", f"RTclass={RTC / 'RT_constitutive_early.bed'},"
                     f"{RTC / 'RT_variable.bed'},{RTC / 'RT_constitutive_late.bed'}",
     "--contrast-group", "GM12878=GM12878", "--contrast-group", "IMR90=IMR90",
     "--chrx", "--haplotypes", HAP, "--pcdh", "--imprinting",
     "--chrx-track-window", "HCG=20000000,GCH=2500000,CROSS=20000000",
     "--chrx-track-step", "500000", "--chrx-distance-step", "auto",
     "--n-threads", THREADS, "--output-dir", CHRX],
    outputs=[CHRX / f for f in ("chrx_concordance.tsv", "chrx_haplotype_concordance.tsv",
                                "chrx_haplotype_tracks.tsv", "chrx_haplotype_distance.tsv")],
    inputs=[HG38 / "rt_4dn" / "signal_track_flags.txt", LAD / "cLAD.bed"],
    log=LOGS / "chrx_classes.log", dry_run=DRY_RUN)

## Step 5. Each copy's contacts

`haplotype_contacts.py`: per line and copy, contact maps of X and DXZ4, P(s), the share of contacts ≥1 Mb (of all X's contacts, at each element and per gene class), the compartment correlation at the ends of contacts ≥2 Mb (GM12878 compartments, 4DN), DXZ4 crossing, escape against inactivated genes (Tukiainen et al. 2017; RefSeq), and GM12878's cells by quartile of XCI strength. IMR-90 is compared both as bulk data would (by haplotype) and by each cell's own inactive X.

In [ ]:
HC = OUT / "haplotype_contacts"
run([PY, S / "haplotype_contacts.py", "--haplotypes", HAP, "--samples", SAMPLES,
     "--group-column", "group", "--groups", "GM12878,IMR90",
     "--xi", "GM12878=1", "--xi", "IMR90=cells", "--xi-cells", PHASED / "allelic_cells.tsv",
     "--boundaries", XA / "gm12878_tad_boundaries_ctcf.hg38.bed",
     "--tads", XA / "gm12878_tad_boundaries.hg38.bed",
     "--compartments", XA / "gm12878_compartments.hg38.bedGraph",
     "--allelic-snps", ELEM / "allelic_snps.tsv",
     "--concordance-tracks", CHRX / "chrx_haplotype_tracks.tsv",
     "--concordance-distance", CHRX / "chrx_haplotype_distance.tsv",
     "--locus", "chrX=chrX", "--locus", "DXZ4=chrX:105800000-125800000",
     "--genes", XA / "ncbiRefSeqCurated.txt.gz", "--windows", "2000000",
     "--xci-strata", "4", "--element", f"island={ISLANDS}",
     *[a for k, p in ELEMENTS.items() for a in ("--element", f"{k}={p}")],
     "--output-dir", HC],
    outputs=[HC / "haplotype_contacts.tsv", HC / "haplotype_xci_trend_GM12878.tsv"],
    inputs=[XA / "gm12878_compartments.hg38.bedGraph", XA / "ncbiRefSeqCurated.txt.gz"],
    log=LOGS / "haplotype_contacts.log", dry_run=DRY_RUN)

## Step 6. The excess expected from the copies' level difference

Pooling a contact's two copies adds ½δ_a δ_p of concordance per contact, where δ is the Xi − Xa level at each end; `copy_mixing_excess.py` sums it over GM12878's contacts by distance (5-Mb SNP blocks resampled for its interval) for Fig. 6h.

In [ ]:
CME = OUT / "copy_mixing_excess"
run([PY, S / "copy_mixing_excess.py", "--allelic-snps", ELEM / "allelic_snps.tsv",
     "--concordance", CHRX / "chrx_concordance.tsv",
     "--ps", HC / "haplotype_contacts_ps.tsv",
     "--haplotype-concordance", CHRX / "chrx_haplotype_concordance.tsv",
     "--group", "GM12878", "--genome", "hg38", "--n-boot", "500",
     "--plot-format", "pdf", "--output-dir", CME],
    outputs=[CME / "copy_mixing_excess.tsv"], log=LOGS / "copy_mixing_excess.log",
    dry_run=DRY_RUN)

## Step 7. Concordance on each copy

`end_posterior_distance_correlation.py --haplotypes` splits every cell into units (its Xi, its Xa, and the autosomal copies carrying the Xi's and the Xa's haplotype), each treated as a cell of its own, so each contact's non-linked partner comes from the same cell and the same copy (near-copies refused, ends centred on the unit's mean), and repeats it at each kind of element (Fig. 6i).

In [ ]:
UNITS = OUT / "units"
run([PY, S / "end_posterior_distance_correlation.py", *POSTERIORS,
     "--samples", SAMPLES, "--group-column", "group", "--haplotypes", HAP,
     "--xi", "GM12878=1", "--xi", "IMR90=cells", "--xi-cells", PHASED / "allelic_cells.tsv",
     "--genome", "hg38", "--null-type", "link-shuffle", "--null-scope", "cell",
     "--null-exclude", "1000", "--cell-center", "on",
     "--min-sites-gch", "1", "--min-sites-hcg", "1",
     "--unit-element", f"island={ISLANDS}",
     *[a for k, p in ELEMENTS.items() for a in ("--unit-element", f"{k}={p}")],
     "--min-pairs", "50", "--n-threads", THREADS, "--output-dir", UNITS],
    outputs=[UNITS / "end_posterior_units_GM12878.tsv"], log=LOGS / "units.log",
    dry_run=DRY_RUN)

## Step 8. Figure 6

In [ ]:
run([PY, S / "figure6_xci.py", "--contacts", HC,
     "--allelic-summary", ELEM / "allelic_summary.tsv",
     "--copy-mixing", CME, "--units", UNITS,
     "--clonal", "GM12878", "--mosaic", "IMR90", "--output", FIG / "Figure6.pdf"],
    outputs=[FIG / "Figure6.pdf"], log=LOGS / "figure6.log",
    dry_run=DRY_RUN, force=True)
show(FIG / "Figure6.pdf")

## Step 9. Checks against the manuscript

In [ ]:
def checks():
    hc = pd.read_csv(HC / "haplotype_contacts.tsv", sep="\t")
    gm = hc[(hc["group"] == "GM12878") & (hc["copies"] == "Xi|Xa")]

    def contacts(measure, set_, col):
        r = gm[(gm["measure"] == measure) & (gm["set"] == set_)]
        return float(r[col].iloc[0])

    trend = pd.read_csv(HC / "haplotype_xci_trend_GM12878.tsv", sep="\t").set_index(["measure", "set"])
    xi_calls = pd.read_csv(HC / "haplotype_xi_calls_IMR90.tsv", sep="\t")["xi_hap"].value_counts()
    strength = pd.read_csv(HC / "haplotype_xci_cells_GM12878.tsv", sep="\t")["strength"]
    summ = pd.read_csv(ELEM / "allelic_summary.tsv", sep="\t")
    rest = summ[(summ["group"] == "GM12878") & (summ["set"] == "chrX rest")].set_index("context")
    SHARE, COMP = "P(s): share ≥ 1 Mb", "A/B compartments: ends' correlation"

    return compare([
        ("GM12878 cells", "469", (sheet["group"] == "GM12878").sum()),
        ("Fig. 6c: Xi share of contacts ≥1 Mb (%)", "64.5", 100 * contacts(SHARE, "chrX", "copy1")),
        ("Fig. 6c: Xa share of contacts ≥1 Mb (%)", "55.3", 100 * contacts(SHARE, "chrX", "copy2")),
        ("Fig. 6c: Xi − Xa (pp)", "9.2", 100 * contacts(SHARE, "chrX", "difference")),
        ("Fig. 6b: compartment correlation, Xi", "0.17", contacts(COMP, "chrX", "copy1")),
        ("Fig. 6b: compartment correlation, Xa", "0.27", contacts(COMP, "chrX", "copy2")),
        ("Fig. 6d: Xi − Xa methylation, CpG islands (pp)", "23", 100 * rest.loc["HCG_island", "xi_minus_xa"]),
        ("Fig. 6d: Xi − Xa methylation, promoters (pp)", "46", 100 * rest.loc["HCG_promoter", "xi_minus_xa"]),
        ("Fig. 6d: Xi − Xa methylation, enhancers (pp)", "38", 100 * rest.loc["HCG_enhancer", "xi_minus_xa"]),
        ("Fig. 6e: IMR-90 cells with hap1 as Xi", "49", xi_calls.get(1, 0)),
        ("Fig. 6e: IMR-90 cells with hap2 as Xi", "92", xi_calls.get(2, 0)),
        ("Fig. 6f: escape genes, Xi − Xa share ≥1 Mb", "0.090", contacts(SHARE, "chrX escape genes", "difference")),
        ("Fig. 6f: escape − inactivated, P", "0.55", contacts(SHARE, "escape − inactivated genes", "p")),
        ("Fig. 6g: GM12878 cells with an XCI strength", "426", strength.notna().sum()),
        ("Fig. 6g: strength trend, contacts ≥1 Mb, P", "0.90", trend.loc[(SHARE, "chrX"), "p_perm"]),
        ("Fig. 6g: strength trend, compartments, P", "0.51", trend.loc[(COMP, "chrX"), "p_perm"]),
    ])


print("dry run: nothing to check yet") if DRY_RUN else checks()